In [ ]:
import pandas as pd

train = pd.read_csv("data/train.csv", parse_dates=["timestamp"])
buildings = pd.read_csv("data/building_metadata.csv")
weather = pd.read_csv("data/weather_train.csv", parse_dates=["timestamp"])

print(train.shape, buildings.shape, weather.shape)
train.head()

(20216100, 4) (1449, 6) (139773, 9)


,building_id,meter,timestamp,meter_reading
0,0,0,2016-01-01,0.0
1,1,0,2016-01-01,0.0
2,2,0,2016-01-01,0.0
3,3,0,2016-01-01,0.0
4,4,0,2016-01-01,0.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# List files in the workspace to locate the uploaded documentation
all_files = []
for root, dirs, files in os.walk('/content'):
    # Skip hidden config folders and sample data to keep output clean
    if '.config' in root or 'sample_data' in root or 'drive' in root:
        continue
    for file in files:
        full_path = os.path.join(root, file)
        all_files.append(full_path)

print("Found files:")
for f in all_files:
    print(f)

Found files:


In [ ]:
import pandas as pd

try:
    metrics_df = pd.read_csv('/metrics.csv')
    print("--- metrics.csv ---")
    display(metrics_df.head())
except Exception as e:
    print("Error loading /metrics.csv:", e)

try:
    results_core_df = pd.read_csv('/results_core.csv')
    print("\n--- results_core.csv ---")
    display(results_core_df.head())
except Exception as e:
    print("Error loading /results_core.csv:", e)

--- metrics.csv ---


,building_id,model_rmse,model_mape,baseline_rmse,baseline_mape
0,569,39.538476,8.736380,40.450708,7.527909
1,571,13.859894,32.379170,14.797960,23.777964
2,576,16.087248,7.042222,14.914371,5.673690
3,577,11.945534,18.109088,17.313299,24.428751
4,588,6.201579,24.734033,10.405556,24.993916



--- results_core.csv ---


,building_id,meter,timestamp,meter_reading,site_id,primary_use,square_feet,year_built,floor_count,air_temperature,...,is_holiday,rolling_24hr_mean,rolling_7day_mean,target_24hr_ahead,actual,predicted,abs_pct_error,sq_error,baseline_sq_error,baseline_abs_pct_error
0,569,0,2016-10-10 00:00:00,283.25,4,Education,86091,1964.0,8.0,22.2,...,1,298.989583,312.976613,284.50,284.50,295.14642,3.742152,113.346330,1.5625,0.439367
1,569,0,2016-10-10 01:00:00,277.50,4,Education,86091,1964.0,8.0,20.6,...,1,298.302083,313.015304,283.50,283.50,291.63360,2.868997,66.155546,36.0000,2.116402
2,569,0,2016-10-10 02:00:00,281.00,4,Education,86091,1964.0,8.0,19.4,...,1,297.614583,313.030185,285.75,285.75,284.54410,0.422013,1.454200,22.5625,1.662292
3,569,0,2016-10-10 03:00:00,278.25,4,Education,86091,1964.0,8.0,17.8,...,1,296.677083,313.036137,281.75,281.75,282.95108,0.426293,1.442594,12.2500,1.242236
4,569,0,2016-10-10 04:00:00,281.50,4,Education,86091,1964.0,8.0,16.1,...,1,296.416667,313.058458,282.50,282.50,286.00443,1.240504,12.280995,1.0000,0.353982


### 6. Interactive Evaluation Dashboard

This dashboard allows you to select any of the 17 core buildings to visualize its actual energy consumption vs. the XGBoost model predictions and the persistence baseline. It also presents building-specific error metrics (RMSE, MAPE).

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from ipywidgets import interact, widgets

# Load the saved results and metrics from root folder
results_core = pd.read_csv('/results_core.csv')
metrics_df = pd.read_csv('/metrics.csv')

# Ensure timestamp is in datetime format
results_core['timestamp'] = pd.to_datetime(results_core['timestamp'])

# Get the list of unique building IDs
building_list = sorted(results_core['building_id'].unique())

@interact(building_id=widgets.Dropdown(options=building_list, description='Building ID:', value=building_list[0]))
def plot_building_predictions(building_id):
    # Filter data
    b_data = results_core[results_core['building_id'] == building_id].sort_values('timestamp')
    b_metrics = metrics_df[metrics_df['building_id'] == building_id].iloc[0]

    # Display local metrics
    print(f"=== Building {building_id} Performance Metrics ===")
    print(f"Model     -> RMSE: {b_metrics['model_rmse']:.2f} | MAPE: {b_metrics['model_mape']:.2f}%")
    print(f"Baseline  -> RMSE: {b_metrics['baseline_rmse']:.2f} | MAPE: {b_metrics['baseline_mape']:.2f}%")
    print("==================================================")

    # Plotly interactive chart
    fig = go.Figure()

    # Actual reading
    fig.add_trace(go.Scatter(
        x=b_data['timestamp'],
        y=b_data['actual'],
        mode='lines',
        name='Actual Energy Consumption',
        line=dict(color='black', width=2)
    ))

    # Model prediction
    fig.add_trace(go.Scatter(
        x=b_data['timestamp'],
        y=b_data['predicted'],
        mode='lines',
        name='XGBoost 24hr Prediction',
        line=dict(color='royalblue', width=1.5, dash='dash')
    ))

    # Baseline prediction
    fig.add_trace(go.Scatter(
        x=b_data['timestamp'],
        y=b_data['meter_reading'], # lag_24h represents yesterday's consumption used as baseline
        mode='lines',
        name='Persistence Baseline (24h Lag)',
        line=dict(color='orange', width=1.2, dash='dot')
    ))

    fig.update_layout(
        title=f"Actual vs. Predicted Energy Consumption for Building {building_id} (Test Set)",
        xaxis_title="Timestamp",
        yaxis_title="Meter Reading (kWh)",
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        height=500
    )

    fig.show()

interactive(children=(Dropdown(description='Building ID:', options=(np.int64(569), np.int64(571), np.int64(576…

In [ ]:
buildings["primary_use"].value_counts()

primary_use
Education                        549
Office                           279
Entertainment/public assembly    184
Public services                  156
Lodging/residential              147
Other                             25
Healthcare                        23
Parking                           22
Warehouse/storage                 13
Manufacturing/industrial          12
Retail                            11
Services                          10
Technology/science                 6
Food sales and service             5
Utility                            4
Religious worship                  3
Name: count, dtype: int64

In [ ]:
elec = train[train["meter"] == 0]
merged = elec.merge(buildings, on="building_id", how="left")
merged = merged.merge(weather, on=["site_id", "timestamp"], how="left")

edu = merged[merged["primary_use"] == "Education"]
print(edu.shape)
edu.isna().sum()

(4597256, 16)


building_id                 0
meter                       0
timestamp                   0
meter_reading               0
site_id                     0
primary_use                 0
square_feet                 0
year_built            1878936
floor_count           3149191
air_temperature         19734
cloud_coverage        2083745
dew_temperature         20719
precip_depth_1_hr     1280118
sea_level_pressure     518115
wind_direction         295265
wind_speed              28680
dtype: int64

In [ ]:
building_ids = edu["building_id"].unique()
print(len(building_ids))

# check data completeness per building before picking - don't pick blind
completeness = edu.groupby("building_id")["air_temperature"].apply(lambda x: x.notna().mean())
completeness.sort_values(ascending=False).head(30)

537


building_id
29     1.000000
403    1.000000
568    1.000000
593    1.000000
604    1.000000
607    1.000000
577    0.999886
655    0.999886
634    0.999886
646    0.999886
639    0.999886
576    0.999886
571    0.999886
569    0.999886
642    0.999886
643    0.999886
638    0.999886
588    0.999886
605    0.999886
618    0.999886
622    0.999886
629    0.999886
589    0.999886
594    0.999886
595    0.999886
599    0.999886
580    0.999886
624    0.999886
628    0.999886
631    0.999886
Name: air_temperature, dtype: float64

In [ ]:
selected_buildings = completeness.sort_values(ascending=False).head(20).index.tolist()
print(selected_buildings)

subset = edu[edu["building_id"].isin(selected_buildings)]
print(subset.shape)
print(subset["timestamp"].min(), subset["timestamp"].max())


[29, 403, 568, 593, 604, 607, 577, 655, 634, 646, 639, 576, 571, 569, 642, 643, 638, 588, 605, 618]
(144431, 16)
2016-01-01 00:00:00 2016-12-31 23:00:00


In [ ]:
subset[subset["building_id"]==29]["meter_reading"].describe()

count    3811.000000
mean      919.223571
std       391.442884
min         0.000000
25%       703.549000
50%      1032.370000
75%      1144.650000
max      2358.930000
Name: meter_reading, dtype: float64

In [ ]:
b29 = subset[subset["building_id"]==29]
print((b29["meter_reading"]==0).sum(), "zero readings out of", len(b29))

330 zero readings out of 3811


In [ ]:
for bid in [403, 568, 593]:
    b = subset[subset["building_id"]==bid]
    zero_pct = (b["meter_reading"]==0).mean() * 100
    print(f"Building {bid}: mean={b['meter_reading'].mean():.1f}, zero%={zero_pct:.1f}%, n={len(b)}")

Building 403: mean=50.6, zero%=0.0%, n=479
Building 568: mean=5.7, zero%=0.0%, n=3298
Building 593: mean=112.9, zero%=0.0%, n=7183


In [ ]:
for bid in selected_buildings:
    b = subset[subset["building_id"]==bid]
    zero_pct = (b["meter_reading"]==0).mean() * 100
    print(f"Building {bid}: n={len(b)}, mean={b['meter_reading'].mean():.1f}, zero%={zero_pct:.1f}%")

Building 29: n=3811, mean=919.2, zero%=8.7%
Building 403: n=479, mean=50.6, zero%=0.0%
Building 568: n=3298, mean=5.7, zero%=0.0%
Building 593: n=7183, mean=112.9, zero%=0.0%
Building 604: n=1012, mean=1.5, zero%=28.2%
Building 607: n=5686, mean=22.1, zero%=0.0%
Building 577: n=8783, mean=56.1, zero%=0.2%
Building 655: n=8783, mean=141.6, zero%=0.0%
Building 634: n=8783, mean=0.3, zero%=0.0%
Building 646: n=8783, mean=42.4, zero%=0.0%
Building 639: n=8783, mean=8.4, zero%=0.0%
Building 576: n=8783, mean=153.6, zero%=0.0%
Building 571: n=8783, mean=42.1, zero%=0.0%
Building 569: n=8783, mean=327.8, zero%=0.0%
Building 642: n=8783, mean=13.4, zero%=0.0%
Building 643: n=8783, mean=90.0, zero%=0.0%
Building 638: n=8783, mean=3.6, zero%=0.0%
Building 588: n=8783, mean=21.9, zero%=0.0%
Building 605: n=8783, mean=24.2, zero%=0.0%
Building 618: n=8783, mean=119.2, zero%=0.0%


In [ ]:
next_candidates = [622, 629, 589, 594, 595, 599, 580, 624, 628, 631]
for bid in next_candidates:
    b = edu[edu["building_id"]==bid]
    zero_pct = (b["meter_reading"]==0).mean() * 100
    print(f"Building {bid}: n={len(b)}, mean={b['meter_reading'].mean():.1f}, zero%={zero_pct:.1f}%")

Building 622: n=8783, mean=216.5, zero%=0.0%
Building 629: n=8783, mean=18.2, zero%=0.0%
Building 589: n=8783, mean=125.7, zero%=0.0%
Building 594: n=8783, mean=381.5, zero%=0.0%
Building 595: n=8783, mean=232.9, zero%=0.0%
Building 599: n=8783, mean=93.3, zero%=0.0%
Building 580: n=8783, mean=101.7, zero%=0.0%
Building 624: n=8783, mean=94.3, zero%=0.0%
Building 628: n=8783, mean=21.4, zero%=0.0%
Building 631: n=8783, mean=12.6, zero%=0.0%


In [ ]:
selected_buildings = [577, 655, 634, 646, 639, 576, 571, 569, 642, 643,
                       638, 588, 605, 618, 622, 629, 589, 594, 595, 599]

subset = edu[edu["building_id"].isin(selected_buildings)]
subset = subset.sort_values(["building_id", "timestamp"])

print(len(selected_buildings), "buildings")
print(subset.shape)

20 buildings
(175660, 16)


In [ ]:
subset["lag_1hr"] = subset.groupby("building_id")["meter_reading"].shift(1)


In [ ]:
subset["lag_24hr"] = subset.groupby("building_id")["meter_reading"].shift(24)   # same hour, yesterday
subset["lag_168hr"] = subset.groupby("building_id")["meter_reading"].shift(168)  # same hour, last week (24*7)

In [ ]:
subset[["building_id","timestamp","meter_reading","lag_1hr","lag_24hr","lag_168hr"]].head(30)

,building_id,timestamp,meter_reading,lag_1hr,lag_24hr,lag_168hr
560,569,2016-01-01 00:00:00,246.00,NaN,NaN,NaN
1948,569,2016-01-01 01:00:00,243.50,246.00,NaN,NaN
3333,569,2016-01-01 02:00:00,248.50,243.50,NaN,NaN
4720,569,2016-01-01 03:00:00,247.00,248.50,NaN,NaN
6108,569,2016-01-01 04:00:00,240.00,247.00,NaN,NaN
7496,569,2016-01-01 05:00:00,245.00,240.00,NaN,NaN
8883,569,2016-01-01 06:00:00,240.25,245.00,NaN,NaN
10268,569,2016-01-01 07:00:00,244.75,240.25,NaN,NaN
11655,569,2016-01-01 08:00:00,240.75,244.75,NaN,NaN
13042,569,2016-01-01 09:00:00,245.25,240.75,NaN,NaN


In [ ]:
subset["hour"] = subset["timestamp"].dt.hour
subset["day_of_week"] = subset["timestamp"].dt.dayofweek  # 0=Monday, 6=Sunday
subset["is_weekend"] = subset["day_of_week"].isin([5,6]).astype(int)
subset[["timestamp","hour","day_of_week","is_weekend"]].head(10)

,timestamp,hour,day_of_week,is_weekend
560,2016-01-01 00:00:00,0,4,0
1948,2016-01-01 01:00:00,1,4,0
3333,2016-01-01 02:00:00,2,4,0
4720,2016-01-01 03:00:00,3,4,0
6108,2016-01-01 04:00:00,4,4,0
7496,2016-01-01 05:00:00,5,4,0
8883,2016-01-01 06:00:00,6,4,0
10268,2016-01-01 07:00:00,7,4,0
11655,2016-01-01 08:00:00,8,4,0
13042,2016-01-01 09:00:00,9,4,0


In [ ]:
import pandas as pd
from pandas.tseries.holiday import USFederalHolidayCalendar

cal = USFederalHolidayCalendar()
holidays_2016 = cal.holidays(start="2016-01-01", end="2016-12-31")

subset["is_holiday"] = subset["timestamp"].dt.normalize().isin(holidays_2016).astype(int)
subset[subset["timestamp"]=="2016-01-01 00:00:00"][["timestamp","is_holiday"]]

,timestamp,is_holiday
560,2016-01-01,1
562,2016-01-01,1
567,2016-01-01,1
568,2016-01-01,1
579,2016-01-01,1
580,2016-01-01,1
583,2016-01-01,1
584,2016-01-01,1
588,2016-01-01,1
593,2016-01-01,1


In [ ]:
subset["rolling_24hr_mean"] = subset.groupby("building_id")["meter_reading"].transform(lambda x: x.rolling(24).mean())
subset["rolling_7day_mean"] = subset.groupby("building_id")["meter_reading"].transform(lambda x: x.rolling(168).mean())
subset[["building_id","timestamp","meter_reading","rolling_24hr_mean","rolling_7day_mean"]].head(30)

,building_id,timestamp,meter_reading,rolling_24hr_mean,rolling_7day_mean
560,569,2016-01-01 00:00:00,246.00,NaN,NaN
1948,569,2016-01-01 01:00:00,243.50,NaN,NaN
3333,569,2016-01-01 02:00:00,248.50,NaN,NaN
4720,569,2016-01-01 03:00:00,247.00,NaN,NaN
6108,569,2016-01-01 04:00:00,240.00,NaN,NaN
7496,569,2016-01-01 05:00:00,245.00,NaN,NaN
8883,569,2016-01-01 06:00:00,240.25,NaN,NaN
10268,569,2016-01-01 07:00:00,244.75,NaN,NaN
11655,569,2016-01-01 08:00:00,240.75,NaN,NaN
13042,569,2016-01-01 09:00:00,245.25,NaN,NaN


In [ ]:
before = len(subset)
subset = subset.dropna(subset=["lag_1hr","lag_24hr","lag_168hr","rolling_24hr_mean","rolling_7day_mean"])
after = len(subset)
print(f"Dropped {before-after} rows ({(before-after)/before*100:.1f}%), {after} remain")

Dropped 3360 rows (1.9%), 172300 remain


In [ ]:
cutoff = "2016-10-10"

train = subset[subset["timestamp"] < cutoff]
test = subset[subset["timestamp"] >= cutoff]

print(f"Train: {len(train)} rows ({len(train)/len(subset)*100:.1f}%)")
print(f"Test: {len(test)} rows ({len(test)/len(subset)*100:.1f}%)")
print(f"Train range: {train['timestamp'].min()} to {train['timestamp'].max()}")
print(f"Test range: {test['timestamp'].min()} to {test['timestamp'].max()}")

Train: 132460 rows (76.9%)
Test: 39840 rows (23.1%)
Train range: 2016-01-08 00:00:00 to 2016-10-09 23:00:00
Test range: 2016-10-10 00:00:00 to 2016-12-31 23:00:00


In [ ]:
baseline_pred = test["lag_168hr"]
actual = test["meter_reading"]

import numpy as np
baseline_rmse = np.sqrt(((actual - baseline_pred) ** 2).mean())
baseline_mape = (abs((actual - baseline_pred) / actual)).mean() * 100

print(f"Baseline RMSE: {baseline_rmse:.2f}")
print(f"Baseline MAPE: {baseline_mape:.2f}%")

Baseline RMSE: 22.73
Baseline MAPE: 18.37%


In [ ]:
feature_cols = ["lag_1hr", "lag_24hr", "lag_168hr", "rolling_24hr_mean", "rolling_7day_mean",
                "hour", "day_of_week", "is_weekend", "is_holiday",
                "air_temperature", "dew_temperature", "wind_speed", "square_feet"]

target_col = "meter_reading"

train = train.dropna(subset=feature_cols + [target_col])
test = test.dropna(subset=feature_cols + [target_col])

X_train, y_train = train[feature_cols], train[target_col]
X_test, y_test = test[feature_cols], test[target_col]

print(X_train.shape, X_test.shape)

(132420, 13) (39840, 13)


In [ ]:
!pip install xgboost


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\DELL\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [ ]:
import sys
print(sys.executable)

c:\Users\DELL\AppData\Local\Programs\Python\Python313\python.exe


In [ ]:
import sys
!{sys.executable} -m pip install xgboost


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import xgboost as xgb

In [ ]:
import xgboost as xgb

model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

model.fit(X_train, y_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [ ]:
import sys
!{sys.executable} -m pip install scikit-learn


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import xgboost as xgb

model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

model.fit(X_train, y_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [ ]:
y_pred = model.predict(X_test)

model_rmse = np.sqrt(((y_test - y_pred) ** 2).mean())
model_mape = (abs((y_test - y_pred) / y_test)).mean() * 100

print(f"Model RMSE: {model_rmse:.2f}")
print(f"Model MAPE: {model_mape:.2f}%")
print(f"Baseline RMSE: {baseline_rmse:.2f}, Baseline MAPE: {baseline_mape:.2f}%")

Model RMSE: 7.36
Model MAPE: 12.26%
Baseline RMSE: 22.73, Baseline MAPE: 18.37%


In [ ]:
y_pred_train = model.predict(X_train)
train_rmse = np.sqrt(((y_train - y_pred_train) ** 2).mean())
train_mape = (abs((y_train - y_pred_train) / y_train)).mean() * 100

print(f"Train RMSE: {train_rmse:.2f}, Train MAPE: {train_mape:.2f}%")
print(f"Test RMSE: {model_rmse:.2f}, Test MAPE: {model_mape:.2f}%")

Train RMSE: 6.62, Train MAPE: inf%
Test RMSE: 7.36, Test MAPE: 12.26%


In [ ]:
mask_train = y_train != 0
mask_test = y_test != 0

train_mape_fixed = (abs((y_train[mask_train] - y_pred_train[mask_train]) / y_train[mask_train])).mean() * 100
test_mape_fixed = (abs((y_test[mask_test] - y_pred[mask_test]) / y_test[mask_test])).mean() * 100

print(f"Train MAPE (excl. zeros): {train_mape_fixed:.2f}%")
print(f"Test MAPE (excl. zeros): {test_mape_fixed:.2f}%")

Train MAPE (excl. zeros): 10.89%
Test MAPE (excl. zeros): 12.26%


In [ ]:
importance = model.feature_importances_
feat_imp = pd.Series(importance, index=feature_cols).sort_values(ascending=False)
print(feat_imp)

lag_1hr              0.965155
lag_168hr            0.020562
lag_24hr             0.003961
hour                 0.003609
square_feet          0.001436
rolling_24hr_mean    0.001357
day_of_week          0.000970
is_holiday           0.000842
rolling_7day_mean    0.000715
air_temperature      0.000585
wind_speed           0.000450
dew_temperature      0.000357
is_weekend           0.000000
dtype: float32


In [ ]:
subset["target_24hr_ahead"] = subset.groupby("building_id")["meter_reading"].shift(-24)
subset[["building_id","timestamp","meter_reading","target_24hr_ahead"]].head(30)

,building_id,timestamp,meter_reading,target_24hr_ahead
233521,569,2016-01-08 00:00:00,255.750,255.00
234903,569,2016-01-08 01:00:00,258.000,256.75
236283,569,2016-01-08 02:00:00,255.000,255.25
237665,569,2016-01-08 03:00:00,258.250,253.50
239048,569,2016-01-08 04:00:00,256.250,257.00
240430,569,2016-01-08 05:00:00,255.500,250.50
241809,569,2016-01-08 06:00:00,262.500,250.75
243188,569,2016-01-08 07:00:00,267.250,252.50
244568,569,2016-01-08 08:00:00,271.000,253.25
245948,569,2016-01-08 09:00:00,289.000,255.00


In [ ]:
feature_cols_24h = ["lag_24hr", "lag_168hr", "rolling_24hr_mean", "rolling_7day_mean",
                     "hour", "day_of_week", "is_weekend", "is_holiday",
                     "air_temperature", "dew_temperature", "wind_speed", "square_feet"]

target_col_24h = "target_24hr_ahead"

subset_clean = subset.dropna(subset=feature_cols_24h + [target_col_24h])

cutoff = "2016-10-10"
train24 = subset_clean[subset_clean["timestamp"] < cutoff]
test24 = subset_clean[subset_clean["timestamp"] >= cutoff]

X_train24, y_train24 = train24[feature_cols_24h], train24[target_col_24h]
X_test24, y_test24 = test24[feature_cols_24h], test24[target_col_24h]

print(X_train24.shape, X_test24.shape)

(132420, 12) (39360, 12)


In [ ]:
model24 = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

model24.fit(X_train24, y_train24)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [ ]:
y_pred24 = model24.predict(X_test24)

model24_rmse = np.sqrt(((y_test24 - y_pred24) ** 2).mean())

mask24 = y_test24 != 0
model24_mape = (abs((y_test24[mask24] - y_pred24[mask24]) / y_test24[mask24])).mean() * 100

print(f"24hr-ahead Model RMSE: {model24_rmse:.2f}")
print(f"24hr-ahead Model MAPE: {model24_mape:.2f}%")

24hr-ahead Model RMSE: 18.94
24hr-ahead Model MAPE: 37.82%


In [ ]:
baseline24_pred = test24["meter_reading"]
actual24 = y_test24

baseline24_rmse = np.sqrt(((actual24 - baseline24_pred) ** 2).mean())

mask_b24 = actual24 != 0
baseline24_mape = (abs((actual24[mask_b24] - baseline24_pred[mask_b24]) / actual24[mask_b24])).mean() * 100

print(f"24hr Baseline RMSE: {baseline24_rmse:.2f}, MAPE: {baseline24_mape:.2f}%")
print(f"24hr Model RMSE: {model24_rmse:.2f}, MAPE: {model24_mape:.2f}%")

24hr Baseline RMSE: 23.90, MAPE: 18.56%
24hr Model RMSE: 18.94, MAPE: 37.82%


In [ ]:
results = test24.copy()
results["actual"] = y_test24.values
results["predicted"] = y_pred24
results["abs_pct_error"] = abs((results["actual"] - results["predicted"]) / results["actual"].replace(0, np.nan)) * 100

per_building = results.groupby("building_id").agg(
    mean_actual=("actual","mean"),
    mape=("abs_pct_error","mean")
).sort_values("mean_actual")

print(per_building)

             mean_actual        mape
building_id                         
634             0.284395  394.346968
638             3.620857   36.140464
639             8.331207   50.572673
642            12.950688   22.049269
629            18.436408   26.884060
588            20.695923   24.734033
605            24.622373   18.489754
571            43.418991   32.379170
646            44.626337   21.349251
577            53.992176   18.109088
643            84.319592   11.600901
599            88.728257   21.124141
618           118.558186   10.069290
589           128.958852   14.631445
655           139.798491   20.053285
576           178.042674    7.042222
622           218.610616    4.129110
595           230.690183   10.335592
569           314.234973    8.736380
594           391.433885    3.706991


In [ ]:
excluded_buildings = [634, 638, 639]

core_mask = ~test24["building_id"].isin(excluded_buildings)

y_test24_core = y_test24[core_mask.values]
y_pred24_core = y_pred24[core_mask.values]

mask_core = y_test24_core != 0
core_mape = (abs((y_test24_core[mask_core] - y_pred24_core[mask_core]) / y_test24_core[mask_core])).mean() * 100
core_rmse = np.sqrt(((y_test24_core - y_pred24_core) ** 2).mean())

print(f"Core (17 buildings) - RMSE: {core_rmse:.2f}, MAPE: {core_mape:.2f}%")
print(f"Full (20 buildings) - RMSE: {model24_rmse:.2f}, MAPE: {model24_mape:.2f}%")
print(f"24hr Baseline (full 20) - RMSE: {baseline24_rmse:.2f}, MAPE: {baseline24_mape:.2f}%")

Core (17 buildings) - RMSE: 20.50, MAPE: 16.20%
Full (20 buildings) - RMSE: 18.94, MAPE: 37.82%
24hr Baseline (full 20) - RMSE: 23.90, MAPE: 18.56%


In [ ]:
baseline24_pred_core = test24["meter_reading"][core_mask.values]

baseline_core_rmse = np.sqrt(((y_test24_core - baseline24_pred_core) ** 2).mean())
baseline_core_mape = (abs((y_test24_core[mask_core] - baseline24_pred_core[mask_core]) / y_test24_core[mask_core])).mean() * 100

print(f"Core Baseline (17 bldgs) - RMSE: {baseline_core_rmse:.2f}, MAPE: {baseline_core_mape:.2f}%")
print(f"Core Model (17 bldgs)    - RMSE: {core_rmse:.2f}, MAPE: {core_mape:.2f}%")

Core Baseline (17 bldgs) - RMSE: 25.87, MAPE: 16.39%
Core Model (17 bldgs)    - RMSE: 20.50, MAPE: 16.20%
